# Bag-of-Words SL (word-heteroskedastic)

Canonical signal-heteroskedastic study: each semantic word has a deterministic hardness quantile, so the per-prompt noise std varies with the prompt composition while the global `Corr(signal, target) = corr` is preserved.

In [1]:
import polars as pl
import torch

from src import get_repo_base
from src.experiments.bag_of_words.sl import BagOfWordsSLConfig

from src.experiments.bag_of_words.analysis import (
    BagOfWordsAnalysisConfig,
    corr_expr,
    rsq_expr,
    mse_expr,
)

repo_root = get_repo_base()
device = torch.device("cuda:0")

/home/nlyu/Code/maxrl-statistics/src/experiments/bag_of_words/sl.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


## Configure

In [ ]:
config = BagOfWordsSLConfig.get_canonical(
    dataset="word_heteroskedastic",
    dataset_base_folder=repo_root / "artifacts" / "bow-data",
    study_base_folder=repo_root / "artifacts" / "bow-sl-word-het-example",
    corr=0.2,
    aux_words_ratio=0.5,
    train_epochs=2,
)

display(config.visualize())
print(f"Study folder: {config.study_folder}")
print(f"Dataset corr target: {config.data.corr:.4f}")
print(f"SNR halflife (word quantile): {config.data.snr_halflife_in_word_quantile}")
print(f"Backbone lr: {config.optimizer.lr:.3e}")
print(f"Head lr:     {config.optimizer.head_lr:.3e}")

state = config.initialize(device=device)
display(
    state.dataset.token_lengths_plot(
        filter_threshold=config.tokenization.filter_samples_above_n_tokens,
    )
)

In [3]:
state.run_training()

sl epoch 0:   0%|          | 0/781 [00:00<?, ?it/s]

validation epoch 0:   0%|          | 0/390 [00:00<?, ?it/s]

sl epoch 1:   0%|          | 0/781 [00:00<?, ?it/s]

validation epoch 1:   0%|          | 0/390 [00:00<?, ?it/s]

## Results

Training saves:
1. A compact `metrics.parquet` to disk which contains per-epoch sufficient statistics to compute metrics.
2. Validation parquet containing per-row ground-truth and target.

In [4]:
metrics_path = config.study_folder / "metrics.parquet"
metrics = pl.read_parquet(metrics_path)
metrics

epoch,train_target_xx,train_target_xy,train_target_yy,train_target_n,train_ground_truth_xx,train_ground_truth_xy,train_ground_truth_yy,train_ground_truth_n,val_target_xx,val_target_xy,val_target_yy,val_target_n,val_ground_truth_xx,val_ground_truth_xy,val_ground_truth_yy,val_ground_truth_n
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,9399.954102,860.531921,49951.992188,49984.0,9399.954102,838.662354,2003.821899,49984.0,3846.845215,1829.383545,50076.695312,49920.0,3846.845215,1755.882568,2002.146606,49920.0
1,4184.89209,1686.746582,49933.089844,49984.0,4184.89209,1491.874023,2003.977051,49984.0,3563.895264,1538.300903,50076.695312,49920.0,3563.895264,1470.524048,2002.146606,49920.0


In [ ]:
analysis = BagOfWordsAnalysisConfig.from_grouped({
    "example": [(0, config.study_folder)]
})

analysis.plot_vs_epoch([
    rsq_expr(split="train", y="ground_truth"),
    rsq_expr(split="val", y="ground_truth"),
    corr_expr(split="train", y="ground_truth"),
    corr_expr(split="val", y="ground_truth"),
])

In [6]:
last_epoch = int(metrics["epoch"].max())
validation_path = config.study_folder / str(last_epoch) / "validation.parquet"
validation_df = pl.read_parquet(validation_path)
validation_df.head()

model_preds,ground_truth,target
f64,f64,f64
-0.05957,-0.367188,-0.098633
0.333984,0.0859375,-0.933594
0.243164,0.03125,0.691406
0.135742,0.0,0.096191
0.139648,0.015625,-0.429688
